In [1]:
# 8_cluster_embeddings_k_means.ipynb
#
# Clusters UKHLS respondents using their semantic embedding vectors (step 6),
# weighted by each respondent's SIPHER population row count (step 7).
# Appends the cluster label to the step-7 table.
#
# Inputs:
#   data/6_add_vector_embedding/k_vector_embedding.pkl         — pidp, nl_profile, embedding
#   data/7_add_sipher_pop_row_counts/k_with_sipher_row_counts.pkl  — step-7 table (includes n_sipher_rows)
#
# Output (data/8_cluster_embeddings_k_means/):
#   k_with_embedding_clusters.pkl / .csv
#   — step-7 table with column `embedding_k_means_cluster` appended

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_variables as _cv
importlib.reload(_cv)
from data_pipeline.config_variables import DATA_FOLDER

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import normalize

import data_pipeline.helpers.cluster as cf
importlib.reload(cf)

from data_pipeline.config_cluster import N_CLUSTERS

# ── Config ────────────────────────────────────────────────────────────────────
EMBED_PKL  = Path(f"../{DATA_FOLDER}/6_add_vector_embedding/k_vector_embedding.pkl")
INPUT_PKL  = Path(f"../{DATA_FOLDER}/7_add_sipher_pop_row_counts/k_with_sipher_row_counts.pkl")
OUT_DIR    = Path(f"../{DATA_FOLDER}/8_cluster_embeddings_k_means")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PKL    = OUT_DIR / "k_with_embedding_clusters.pkl"
OUT_CSV    = OUT_DIR / "k_with_embedding_clusters.csv"

for p in (EMBED_PKL, INPUT_PKL):
    if not p.exists():
        raise FileNotFoundError(f"{p} not found")

# ── 1. Load step-7 table (source of truth + weights) ─────────────────────────
print(f"Reading {INPUT_PKL.name} ...")
df = pd.read_pickle(INPUT_PKL)
df["pidp"] = pd.to_numeric(df["pidp"], errors="coerce").astype("int64")
print(f"  {len(df):,} respondents × {len(df.columns)} cols")

# ── 2. Load embeddings ────────────────────────────────────────────────────────
print(f"\nReading {EMBED_PKL.name} ...")
df_emb = pd.read_pickle(EMBED_PKL)
df_emb["pidp"] = pd.to_numeric(df_emb["pidp"], errors="coerce").astype("int64")
print(f"  {len(df_emb):,} respondents with embeddings")

# Stack list-valued embedding column into a 2-D matrix
X = np.array(df_emb["embedding"].tolist(), dtype=np.float32)
print(f"  Embedding matrix: {X.shape}")

# L2-normalise (standard for cosine-distance clustering)
Xn = normalize(X, norm="l2", axis=1)

# ── 3. Align SIPHER weights to embedding rows ─────────────────────────────────
weight_map = df.set_index("pidp")["n_sipher_rows"]
weights = df_emb["pidp"].map(weight_map).fillna(0).astype(np.float64).values
# Fall back to uniform if all weights are zero
if weights.sum() == 0:
    print("  WARNING: no n_sipher_rows matched — using uniform weights")
    weights = None
else:
    print(f"  Weights: min={weights.min():.0f}  max={weights.max():.0f}  "
          f"mean={weights.mean():.1f}  zero={int((weights == 0).sum())}")

# ── 4. Fit K-Means ───────────────────────────────────────────────────────────
n        = len(Xn)
labels   = cf.fit_kmeans(Xn, N_CLUSTERS, sample_weight=weights)
print(f"\n  {n:,} respondents → {N_CLUSTERS} embedding cluster(s)")

df_emb["embedding_k_means_cluster"] = labels + 1   # 1-indexed

# ── 5. Append cluster column to step-7 table ─────────────────────────────────
df = df.merge(
    df_emb[["pidp", "embedding_k_means_cluster"]],
    on="pidp",
    how="left",
)
df["embedding_k_means_cluster"] = df["embedding_k_means_cluster"].fillna(0).astype("int64")

n_matched = (df["embedding_k_means_cluster"] > 0).sum()
print(f"  {n_matched:,} / {len(df):,} respondents assigned to a cluster")
print(f"  Cluster distribution:\n{df['embedding_k_means_cluster'].value_counts().sort_index().to_string()}")

# ── 6. Save ───────────────────────────────────────────────────────────────────
df.to_pickle(OUT_PKL)
df.to_csv(OUT_CSV, index=False)
print(f"\nSaved {len(df):,} rows × {len(df.columns)} cols")
print(f"  → {OUT_PKL}")
print(f"  → {OUT_CSV}")


Reading k_with_sipher_row_counts.pkl ...
  27,330 respondents × 68 cols

Reading k_vector_embedding.pkl ...
  27,330 respondents with embeddings
  Embedding matrix: (27330, 1536)
  Weights: min=255  max=43858  mean=1933.9  zero=0

  27,330 respondents → 10 embedding cluster(s)
  27,330 / 27,330 respondents assigned to a cluster
  Cluster distribution:
embedding_k_means_cluster
1     4042
2     3651
3     2301
4     3440
5     2847
6     1346
7     3090
8     1247
9     4438
10     928

Saved 27,330 rows × 69 cols
  → ../data/8_cluster_embeddings_k_means/k_with_embedding_clusters.pkl
  → ../data/8_cluster_embeddings_k_means/k_with_embedding_clusters.csv


In [2]:
# ── 7. Export API summary CSV ─────────────────────────────────────────────────
# Writes a pre-aggregated per-cluster summary to api/data/clusters/ so the API
# can serve it directly without loading the full PKL on every request.

import data_pipeline.helpers.cluster_summary as _cs
importlib.reload(_cs)
from data_pipeline.config_cluster import WAVE as _WAVE

_summary = _cs.make_cluster_summary(df, "embedding_k_means_cluster", wave=_WAVE)

_api_clusters_dir = Path("../api/data/clusters")
_api_clusters_dir.mkdir(parents=True, exist_ok=True)
_summary_csv = _api_clusters_dir / "national_embedding_clusters.csv"
_summary.to_csv(_summary_csv, index=False)

print(f"API summary → {_summary_csv}")
print(_summary.to_string(index=False))


API summary → ../api/data/clusters/national_embedding_clusters.csv
 cluster_id tribe_label    size  n_respondents  age  health sex_dv  sex_dv_pct             jbstat  jbstat_pct                racel_dv  racel_dv_pct        hiqual_dv  hiqual_dv_pct            marstat_dv  marstat_dv_pct      tenure_dv  tenure_dv_pct                              hhtype_dv  hhtype_dv_pct
          1   Cluster 1 6511146           4042 79.4     3.0 Female         100            Retired          92                   White           100 No qualification             21 Married/Civil partner              55 Owner-occupied             84                    Couple, no children             53
          2   Cluster 2 5963153           3651 78.7     3.0   Male         100            Retired          84                   White           100           Degree             26 Married/Civil partner              71 Owner-occupied             86                    Couple, no children             66
          3   Cluster 3 642